In [1]:
import hashlib
import re
import unicodedata
from pathlib import Path

# 👇 Pega aquí el hash que te muestra el servidor (ojo: puede cambiar si reconectas)
TARGET = "5ec47d9ff2a0a77afa812d7b2d3623af2c4c8c74447f5e1d1d77488379fcc475"

# 👇 Ruta a tu cheese_list.txt (ajústala si está en otro lado)
CHEESE_LIST_PATH = CHEESE_LIST_PATH = "cheese_list.txt"

def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def normalize_variants(s: str) -> list[str]:
    """
    Genera variantes típicas de normalización que un reto puede aplicar.
    """
    s = s.strip()

    # Quitar acentos (por si hay nombres con tildes en la lista)
    s_no_acc = "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

    # Solo alfanumérico (quita espacios, comillas, etc.)
    s_alnum = re.sub(r"[^A-Za-z0-9]", "", s_no_acc)

    variants = [
        s,
        s.lower(),
        s.upper(),
        s.capitalize(),

        s_no_acc,
        s_no_acc.lower(),

        s.replace(" ", ""),
        s.lower().replace(" ", ""),
        s_no_acc.replace(" ", ""),
        s_no_acc.lower().replace(" ", ""),

        s.replace(" ", "_"),
        s.lower().replace(" ", "_"),
        s.replace(" ", "-"),
        s.lower().replace(" ", "-"),

        s_alnum,
        s_alnum.lower(),
        s_alnum.upper(),
    ]

    # Quitar duplicados manteniendo orden
    seen = set()
    out = []
    for v in variants:
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out


def try_all(cheese: str) -> tuple[str, str, str] | None:
    """
    Devuelve (cheese_variant, salt_hex, mode) si encuentra match.
    """
    for v in normalize_variants(cheese):
        vb = v.encode("utf-8", errors="ignore")

        for i in range(256):
            salt_hex = format(i, "02x")          # "00".."ff"
            salt_ascii = salt_hex.encode("ascii") # b"0f"
            salt_byte = bytes.fromhex(salt_hex)   # b"\x0f"

            # Combinaciones comunes (ASCII salt)
            combos = [
                (vb + salt_ascii, "cheese + salt_ascii"),
                (salt_ascii + vb, "salt_ascii + cheese"),
                (vb + b":" + salt_ascii, "cheese : salt_ascii"),
                (vb + b"|" + salt_ascii, "cheese | salt_ascii"),
                (salt_ascii + b":" + vb, "salt_ascii : cheese"),
            ]

            # Combinaciones comunes (byte salt)
            combos += [
                (vb + salt_byte, "cheese + salt_byte"),
                (salt_byte + vb, "salt_byte + cheese"),
            ]

            for data, mode in combos:
                if sha256_hex(data) == TARGET:
                    return v, salt_hex, mode

    return None


def main():
    p = Path(CHEESE_LIST_PATH)
    if not p.exists():
        print(f"[!] No existe el archivo: {p}")
        return

    print("[*] Loading cheeses...")
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        cheeses = [line.strip("\r\n") for line in f if line.strip()]

    print(f"[*] Cheeses loaded: {len(cheeses)}")
    print("[*] Cracking...")

    for idx, cheese in enumerate(cheeses, start=1):
        res = try_all(cheese)
        if res:
            v, salt_hex, mode = res
            print("\n✅ FOUND!")
            print("Original cheese line:", repr(cheese))
            print("Used cheese variant :", repr(v))
            print("Salt (hex)          :", salt_hex)
            print("Mode                :", mode)
            print("\n➡️ En el nc, prueba responder exactamente con el 'Used cheese variant' + el salt según el modo.")
            return

        # progreso cada 100
        if idx % 100 == 0:
            print(f"  ...checked {idx}/{len(cheeses)}")

    print("\n❌ Not found (con estas variantes).")


if __name__ == "__main__":
    main()

[*] Loading cheeses...
[*] Cheeses loaded: 599
[*] Cracking...
  ...checked 100/599
  ...checked 200/599

✅ FOUND!
Original cheese line: 'Toscanello'
Used cheese variant : 'toscanello'
Salt (hex)          : 8d
Mode                : cheese + salt_byte

➡️ En el nc, prueba responder exactamente con el 'Used cheese variant' + el salt según el modo.
